In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")
from main import GPTModel

/Users/himmat/LLM-from-Scratch/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [3]:
torch.manual_seed(123)

In [4]:
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [5]:
import tiktoken

def text_to_token(text, tokenizer):
    encode = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encode).unsqueeze(0) # Add a batch dimensions
    return encoded_tensor

In [6]:
context = "Hello, how are you"
tokenizer = tiktoken.get_encoding('gpt2')
token = text_to_token(context, tokenizer)
token

tensor([[15496,    11,   703,   389,   345]])

In [7]:
def token_to_text(token, tokenizer):
    decode = token.squeeze(0) # Removing batch dimensions
    text = tokenizer.decode(decode.tolist()) # converting tensors into list
    return text

In [8]:
token_to_text(token, tokenizer)

'Hello, how are you'

In [9]:
from main import GenerateSimpleText

token_ids = GenerateSimpleText(model, text_to_token(context, tokenizer), max_num_token=10, context_size=256)
token_ids.squeeze(0)

tensor([15496,    11,   703,   389,   345, 30057,  6287,  5286, 46073, 17822,
        34494, 12250, 32013,  7165, 34341])

In [10]:
token_to_text(token_ids, tokenizer)

'Hello, how are you261 extentaded EngelLab shave neighbour finalized crazy Roots'

## Calculating text generation loss: cross-entropy and perplexity

In [11]:
with open("the-verdict.txt", 'r') as f:
    text_data = f.read()

In [12]:
print(len(text_data))
print(len(tokenizer.encode(text_data)))

20479
5145


In [13]:
train_ration = 0.9
split = int(train_ration * len(text_data))
train_data = text_data[:split]
test_data = text_data[split:]

In [14]:
from main import create_dataloader_v1

In [15]:
torch.manual_seed(123)

In [16]:
train_data_loader = create_dataloader_v1(
    train_data, 2, GPT_CONFIG_124M['context_length'], stride=GPT_CONFIG_124M['context_length'], shuffle=True, drop_last=True, num_workers=0
)

val_data_loader = create_dataloader_v1(
    test_data, 2, GPT_CONFIG_124M['context_length'], stride=GPT_CONFIG_124M['context_length'], shuffle=False, drop_last=False, num_workers=0
)

In [17]:
for x, y in train_data_loader:
    print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [18]:
train_token = 0
for x, y in train_data_loader:
    train_token += x.numel()

test_token = 0
for x, y in val_data_loader:
    test_token += x.numel()

print("Train Token", train_token)
print("Test token", test_token)
print("Total token", train_token + test_token)

Train Token 4608
Test token 512
Total token 5120


In [19]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float('nan')
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [20]:
with torch.no_grad():
    training_loss = calc_loss_loader(train_data_loader, model, device="cpu")
    val_loss = calc_loss_loader(val_data_loader, model, device="cpu")

print("Training Loss: ", training_loss)
print("Val Loss: ", val_loss)

Training Loss:  10.987385325961643
Val Loss:  10.980905532836914
